# 03c — REHAB24-6: expanding session statistics (cells for `benchmark_3d_world.ipynb` on Colab)

| name | bone lengths and scale come from | length unit |
|---|---|---|
| `window` (Phase 2) | the last 300 frames (~10 s); older frames are forgotten | shoulder width |
| `exp_sw` | **every frame since the session started** | shoulder width |
| `exp_torso` | every frame since the session started | **torso length** (mid-shoulder to mid-hip, as in `pose_normalise.py`); shoulder width until the hips have been seen |

No pose is asked of the patient: the session estimates simply keep improving as they exercise.

**Order**
1. Push the repo with the Phase 4A `iml_live.py` to GitHub.
2. Colab: Runtime → Restart, then STEP 1 → STEP 5 of `benchmark_3d_world.ipynb` (cached landmarks, quick).
3. Paste **cell S3** after STEP 5 and run it (about twice as long as cell S1: two variants).
4. Paste **cell S4** and run it → `RESULTS/feat_rehab24_stream_v2.npz`.
5. Download it into `data/fit3d/` next to `feat_rehab24_stream.npz` (the Phase 2 file is still needed).

In [ ]:
# ===== CELL S3 - replay with expanding statistics, shoulder and torso scale =====
import time, importlib
import numpy as np
import iml_live; importlib.reload(iml_live)
from iml_live import stream_features
from features import overlap_rows
import inspect
assert 'expanding' in inspect.getsource(iml_live.FeatureStream), 'old iml_live.py - push Phase 4A and re-run STEP 2'

rec_by_id = {r['rec_id']: r for r in recordings}
by_rec = {}
for it in aligned_cache:
    rid, side = it['meta']['recording'].rsplit('_', 1)
    by_rec.setdefault(rid, []).append((side, it))

VARIANTS = {'exp_sw': dict(session='expanding', scale='shoulder'),
            'exp_torso': dict(session='expanding', scale='torso')}
t0, bad = time.time(), []
for k, (rid, items) in enumerate(by_rec.items(), 1):
    lm, world, size, fps = get_3d_landmarks(rec_by_id[rid])
    for side, it in items:
        for name, kw in VARIANTS.items():
            S = stream_features(lm, world, side, size, fps, threshold=THRESHOLD, **kw)
            S_al, _ = overlap_rows(S, np.zeros(len(world)), it['meta']['lag'])
            if S_al.shape != np.asarray(it['feat']).shape:
                bad.append((it['meta']['recording'], name)); continue
            it['feat_' + name] = S_al
    print(f'[{k}/{len(by_rec)}] {rid}  {time.time() - t0:.0f}s', flush=True)
print(f'\ndone, {len(bad)} shape mismatches')

In [ ]:
# ===== CELL S4 - save (same frames, same order as feat_rehab24_stream.npz) =====
import os
from features import FEATURE_NAMES
I2D = FEATURE_NAMES.index('angle_2d')
parts = []
for it in aligned_cache:
    if 'feat_exp_sw' not in it or 'feat_exp_torso' not in it:
        continue
    F, A, B, t = (np.asarray(it[k], float) for k in ('feat', 'feat_exp_sw', 'feat_exp_torso', 'truth_3d'))
    n = min(len(F), len(A), len(B), len(t)); F, A, B, t = F[:n], A[:n], B[:n], t[:n]
    ok = np.isfinite(t) & (np.isfinite(F[:, I2D]) | np.isfinite(A[:, I2D]))   # same rule as cell S2
    if ok.any():
        parts.append((A[ok], B[ok], t[ok], it['meta']))
n = [len(p[2]) for p in parts]
out = os.path.join(RESULTS, 'feat_rehab24_stream_v2.npz')
np.savez_compressed(out,
    X_exp_sw=np.concatenate([p[0] for p in parts]).astype(np.float32),
    X_exp_torso=np.concatenate([p[1] for p in parts]).astype(np.float32),
    y=np.concatenate([p[2] for p in parts]).astype(np.float32),
    recording=np.repeat(['rehab_' + str(p[3]['recording']) for p in parts], n),
    feature_names=np.array(FEATURE_NAMES))
print(f'saved {out}: {sum(n):,} frames, {len(parts)} arm-recordings')